# Task 1 - Environement Setup & Data Ingesion

### Connect Google Drive

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Configure PySpark in Colab

In [ ]:
! pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigDataAssignment") \
    .getOrCreate()

spark

### Load the Dataset

In [ ]:
# Loading the dataset into PySpark
df = spark.read.csv(
    "/content/drive/MyDrive/Big Data Assignment - Group 02/online_retail_II.xlsx",
    header=True,
    inferSchema=True
)

# Displaying the First 5 Rows
df.show(5)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|"RECORD FRAME 7""...|      48|2009-12-01 07:45:00|  2.1|    13085.0|United Kingdom|
| 489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
only showing top 5 rows


### Display Schema and Record Counts

In [ ]:
df.printSchema()
df.count()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



525461

### Create Bronze Layer

In [ ]:
from pyspark.sql.functions import year, month

bronze_df = df.withColumn(
    "year", year("InvoiceDate")
).withColumn(
    "month", month("InvoiceDate")
)

In [ ]:
silver_df = spark.read.parquet("/content/bronze")

silver_df = silver_df.withColumnRenamed("Customer ID", "CustomerID") \
                     .withColumnRenamed("Invoice", "InvoiceNo") \
                     .withColumnRenamed("Price", "UnitPrice")

### Store Bronze Data as Partitioned Parquet

In [ ]:
bronze_df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("/content/bronze")

# Task 2 - Data Cleaning and Quality Management

In [ ]:
from pyspark.sql.functions import col, sum, when

# Initial count
initial_count = df.count()

### Handling Missing CustomerID

In [ ]:
missing_customer = df.filter(col("Customer ID").isNull())
missing_customer_count = missing_customer.count()
df1 = df.filter(col("Customer ID").isNotNull())

### Negetive Quantities (returns)

In [ ]:
negative_qty = df1.filter(col("Quantity") <= 0)
negative_qty_count = negative_qty.count()
df2 = df1.filter(col("Quantity") > 0)

### Cancelled Invoices

In [ ]:
cancelled = df2.filter(col("Invoice").startswith("C"))
cancelled_count = cancelled.count()
df3 = df2.filter(~col("Invoice").startswith("C"))

### Invalid or extreme prices

In [ ]:
invalid_price = df3.filter(col("Price") <= 0)
invalid_price_count = invalid_price.count()
df4 = df3.filter(col("Price") > 0)

### Duplicate Records

In [ ]:
before_dupes = df4.count()
df_clean = df4.dropDuplicates()
duplicate_count = before_dupes - df_clean.count()

### Null Counts Per Column

In [ ]:
from pyspark.sql.functions import sum, when

null_counts = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
null_counts.show()

+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|Customer ID|Country|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|      0|        0|       2928|       0|          0|    0|     107927|      0|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+



### Data Quality Report

In [ ]:
# Final Count
final_count = df_clean.count()

# Data Quality Report
data_quality_report = spark.createDataFrame([
    ("Total Records Ingested", initial_count),
    ("Missing CustomerID Removed", missing_customer_count),
    ("Negative Quantity Removed", negative_qty_count),
    ("Cancelled Invoices Removed", cancelled_count),
    ("Invalid Price Removed", invalid_price_count),
    ("Duplicate Records Removed", duplicate_count),
    ("Final Clean Records", final_count)
], ["Metric", "Count"])

data_quality_report.show()

+--------------------+------+
|              Metric| Count|
+--------------------+------+
|Total Records Ing...|525461|
|Missing CustomerI...|107927|
|Negative Quantity...|  9839|
|Cancelled Invoice...|     0|
|Invalid Price Rem...|    31|
|Duplicate Records...|  6748|
| Final Clean Records|400916|
+--------------------+------+

